# Lab 2A · Detection by hand

**Computer Vision for Developers · SDAIA Academy** · Day 2 · Blocks 2 and 4 · Acts 2, 4, 6 and 7

**Goal:** Write IoU and NMS yourself and match torchvision.ops, compute precision, recall and AP by hand (≈ 0.68 on the slide example), then run Faster R-CNN, Mask R-CNN and Keypoint R-CNN and read their outputs field by field.

## How to use this notebook

1. Switch the runtime to a **T4 GPU** first: *Runtime → Change runtime type → T4 GPU*.
2. Run the cells **from top to bottom** with *Shift + Enter*. Do not skip cells: later cells use variables from earlier ones.
3. Cells with a **TODO** comment need **one line** from you. The cell right after each TODO is a check: it prints ✅ when your line is right, or a hint when it is not.
4. Nobody writes a cell from empty. If you are stuck for more than five minutes, open the *solved* version of this notebook and compare.
5. Colab forgets files when the session ends. If you come back later, run everything again from the top.
6. The **Extra tasks** at the end are optional and are not graded.

Expected time: about 40 minutes in class; finish the rest at home.

In [ ]:
# GPU check, with a CPU fallback
import os, torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    print("✅ GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ No GPU found. Runtime → Change runtime type → T4 GPU, then Runtime → Restart.")
    print("   You can keep going on CPU: the notebook switches to the same settings (inference only, a little slower).")
# FAST mode shrinks data and epochs so everything still finishes on a CPU.
FAST = DEVICE == "cpu" or os.environ.get("CVD_FAST") == "1"
print("DEVICE =", DEVICE, "| FAST =", FAST)

In [ ]:
import numpy as np, matplotlib.pyplot as plt, urllib.request
import torchvision
from torchvision.ops import box_iou, nms
from torchvision.io import decode_image
from torchvision.utils import draw_bounding_boxes, draw_segmentation_masks, draw_keypoints
print("torchvision", torchvision.__version__)

## 2 · Describing a box

torchvision uses **xyxy** in pixels: `(x_min, y_min, x_max, y_max)`. COCO uses **xywh**, YOLO uses normalised **cxcywh**. Same box, three ways of writing it (slide 2.18).

In [ ]:
def xyxy_to_xywh(b):  x1, y1, x2, y2 = b; return (x1, y1, x2 - x1, y2 - y1)
def xyxy_to_cxcywh(b): x1, y1, x2, y2 = b; return ((x1 + x2) / 2, (y1 + y2) / 2, x2 - x1, y2 - y1)
box, W, H = (100, 120, 300, 360), 640, 480
cx, cy, w, h = xyxy_to_cxcywh(box)
print("xywh       ", xyxy_to_xywh(box))
print("cxcywh     ", (cx, cy, w, h))
print("normalised ", (cx / W, cy / H, w / W, h / H))

### 2.1 · IoU by hand
Intersection over Union = overlap area ÷ combined area. Fill in the intersection line — remember `max(0, …)` for boxes that do not overlap.

In [ ]:
def iou(a, b):
    ix1, iy1 = max(a[0], b[0]), max(a[1], b[1])
    ix2, iy2 = min(a[2], b[2]), min(a[3], b[3])
    inter = 0  # TODO: intersection area = max(0, ix2 - ix1) * max(0, iy2 - iy1)
    area = lambda r: (r[2] - r[0]) * (r[3] - r[1])
    union = area(a) + area(b) - inter
    return inter / union

print("example 1:", round(iou((0, 0, 4, 4), (2, 2, 6, 6)), 3))   # slide 2.21
print("example 2:", round(iou((0, 0, 4, 4), (1, 0, 5, 4)), 3))   # slide 2.22
print("no overlap:", iou((0, 0, 1, 1), (5, 5, 6, 6)))

In [ ]:
# Check
try:
    assert abs(iou((0,0,4,4),(2,2,6,6)) - 4/28) < 1e-9 and abs(iou((0,0,4,4),(1,0,5,4)) - 0.6) < 1e-9 and iou((0,0,1,1),(5,5,6,6)) == 0
    print('✅ IoU matches both worked examples and handles no overlap.')
except Exception as e:
    print('❌ Not yet. Hint: inter = max(0, ix2 - ix1) * max(0, iy2 - iy1)')

In [ ]:
rng = np.random.default_rng(0)
xy = rng.uniform(0, 400, (20, 2)); wh = rng.uniform(20, 150, (20, 2))
boxes = np.hstack([xy, xy + wh]).astype(np.float32)
mine = np.array([[iou(a, b) for b in boxes] for a in boxes])
lib = box_iou(torch.tensor(boxes), torch.tensor(boxes)).numpy()
print("max difference vs torchvision.ops.box_iou:", float(np.abs(mine - lib).max()))

## 4 · Scale and duplicates: NMS

**NMS**: keep the highest-confidence box, delete every other box that overlaps it by more than an IoU threshold, then repeat with the next highest remaining box. These six boxes reproduce slide 2.48.

In [ ]:
names = list("ABCDEF")
det_boxes = np.array([[100, 100, 200, 200], [110, 105, 210, 205], [300, 100, 400, 200],
                      [90, 120, 190, 215], [310, 110, 405, 210], [500, 300, 600, 400]], dtype=np.float32)
det_scores = np.array([0.95, 0.90, 0.80, 0.75, 0.60, 0.55], dtype=np.float32)
print("IoU(A,B) =", round(iou(det_boxes[0], det_boxes[1]), 2), "| IoU(A,D) =", round(iou(det_boxes[0], det_boxes[3]), 2),
      "| IoU(C,E) =", round(iou(det_boxes[2], det_boxes[4]), 2))

def nms_numpy(boxes, scores, iou_thr=0.5):
    order = np.argsort(-scores)          # highest score first
    keep = []
    while len(order) > 0:
        best = order[0]
        keep.append(int(best))
        ious = np.array([iou(boxes[best], boxes[o]) for o in order[1:]])
        order = order[1:]  # TODO: keep only the remaining boxes whose IoU with best is <= iou_thr
    return keep

kept = nms_numpy(det_boxes, det_scores)
print("kept:", [names[k] for k in kept])

In [ ]:
# Check
try:
    assert kept == [0, 2, 5]
    print('✅ NMS keeps A, C and F, like slide 2.48.')
except Exception as e:
    print('❌ Not yet. Hint: order = order[1:][ious <= iou_thr]')

In [ ]:
scores_r = rng.uniform(0.3, 1.0, len(boxes)).astype(np.float32)
mine = nms_numpy(boxes, scores_r, 0.5)
lib = nms(torch.tensor(boxes), torch.tensor(scores_r), 0.5).tolist()
print("yours:      ", mine)
print("torchvision:", lib)
print("same boxes kept:", mine == lib)

## 6 · How good is the detector? AP by hand

Five ground-truth objects, eight predictions sorted by confidence, already marked TP/FP with IoU ≥ 0.5 (slides 2.74–2.76).

In [ ]:
tps = np.array([1, 1, 0, 1, 0, 1, 0, 0])     # TP TP FP TP FP TP FP FP
n_gt = 5
tp_cum = np.cumsum(tps)
precision = np.zeros(len(tps))  # TODO: TPs so far ÷ predictions so far (np.arange(1, len(tps) + 1))
recall = tp_cum / n_gt
for k in range(len(tps)):
    print(f"{k+1}  {'TP' if tps[k] else 'FP'}  P {precision[k]:.2f}  R {recall[k]:.1f}")

In [ ]:
# Check
try:
    assert np.allclose(precision, [1, 1, 2/3, 3/4, 3/5, 4/6, 4/7, 4/8])
    print('✅ Precision column matches slide 2.75.')
except Exception as e:
    print('❌ Not yet. Hint: precision = tp_cum / np.arange(1, len(tps) + 1)')

In [ ]:
def average_precision(precision, recall):
    """Area under the interpolated PR curve (slide 2.76)."""
    ap, prev_r = 0.0, 0.0
    for k in range(len(recall)):
        if recall[k] > prev_r:                      # recall went up at this prediction
            ap += (recall[k] - prev_r) * precision[k:].max()
            prev_r = recall[k]
    return ap

ap = average_precision(precision, recall)
print(f"AP = {ap:.3f}")
plt.figure(figsize=(4.5, 3.4))
plt.step(np.r_[0, recall], np.r_[1, precision], where="post", marker="o")
plt.xlabel("recall"); plt.ylabel("precision"); plt.xlim(0, 1.02); plt.ylim(0, 1.05); plt.title(f"PR curve · AP = {ap:.2f}")
plt.show()

In [ ]:
# mAP = mean of per-class AP. A second class "bicycle" with 3 objects:
ap_car = ap
tps_b = np.array([1, 0, 0, 1, 0]); cum = np.cumsum(tps_b)
ap_bike = average_precision(cum / np.arange(1, 6), cum / 3)
print(f"AP car {ap_car:.2f} | AP bicycle {ap_bike:.2f} | mAP {(ap_car + ap_bike) / 2:.2f}")

## 3 & 7 · Real detectors from torchvision

All three models share a ResNet-50 + FPN backbone (Day 1) and differ only in their heads. We use the Ultralytics sample photo `bus.jpg`; if the download fails we fall back to a built-in image.

In [ ]:
try:
    urllib.request.urlretrieve("https://ultralytics.com/images/bus.jpg", "bus.jpg")
    img = decode_image("bus.jpg")
except Exception as e:
    print("Download failed, using a built-in image instead:", e)
    from skimage import data
    img = torch.from_numpy(data.astronaut()).permute(2, 0, 1)
print("image tensor:", tuple(img.shape), img.dtype)   # (C, H, W) uint8
plt.imshow(img.permute(1, 2, 0)); plt.axis("off"); plt.show()

In [ ]:
from torchvision.models.detection import fasterrcnn_resnet50_fpn_v2, FasterRCNN_ResNet50_FPN_V2_Weights
weights = FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
frcnn = fasterrcnn_resnet50_fpn_v2(weights=weights, box_score_thresh=0.05).to(DEVICE).eval()   # eval()!
CATS = weights.meta["categories"]
preprocess = weights.transforms()
with torch.no_grad():
    out = frcnn([preprocess(img).to(DEVICE)])[0]
print("fields:", list(out.keys()))
print("boxes", tuple(out["boxes"].shape), "| labels", tuple(out["labels"].shape), "| scores", tuple(out["scores"].shape))
for b, l, s in list(zip(out["boxes"], out["labels"], out["scores"]))[:5]:
    print(f"{CATS[l]:<10} {s:.2f}  xyxy {[round(v) for v in b.tolist()]}")

In [ ]:
CONF = 0.5
keep = torch.ones_like(out["scores"], dtype=torch.bool)  # TODO: keep only boxes whose score is above CONF
labels = [f"{CATS[l]} {s:.2f}" for l, s in zip(out["labels"][keep], out["scores"][keep])]
drawn = draw_bounding_boxes(img, out["boxes"][keep].cpu(), labels, width=4, colors="lime", font_size=24)
print(int(keep.sum()), "boxes above", CONF)
plt.figure(figsize=(6, 8)); plt.imshow(drawn.permute(1, 2, 0)); plt.axis("off"); plt.show()

In [ ]:
# Check
try:
    assert keep.dtype == torch.bool and int(keep.sum()) == int((out["scores"] > CONF).sum())
    print('✅ Confidence filter is correct.')
except Exception as e:
    print('❌ Not yet. Hint: keep = out["scores"] > CONF')

In [ ]:
from torchvision.models.detection import maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights
mweights = MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT
mrcnn = maskrcnn_resnet50_fpn_v2(weights=mweights).to(DEVICE).eval()
with torch.no_grad():
    mout = mrcnn([mweights.transforms()(img).to(DEVICE)])[0]
print("fields:", list(mout.keys()), "| masks", tuple(mout["masks"].shape), "(N, 1, H, W) probabilities")
mk = mout["scores"] > 0.5
masks = (mout["masks"][mk, 0] > 0.5).cpu()
seg = draw_segmentation_masks(img, masks, alpha=0.55)
plt.figure(figsize=(6, 8)); plt.imshow(seg.permute(1, 2, 0)); plt.axis("off"); plt.show()

In [ ]:
from torchvision.models.detection import keypointrcnn_resnet50_fpn, KeypointRCNN_ResNet50_FPN_Weights
kweights = KeypointRCNN_ResNet50_FPN_Weights.DEFAULT
krcnn = keypointrcnn_resnet50_fpn(weights=kweights).to(DEVICE).eval()
with torch.no_grad():
    kout = krcnn([kweights.transforms()(img).to(DEVICE)])[0]
print("keypoints", tuple(kout["keypoints"].shape), "→ (N people, 17 joints, [x, y, visibility])")
print("joint names:", kweights.meta["keypoint_names"][:5], "…")
kk = kout["scores"] > 0.8
SKELETON = [(0, 1), (0, 2), (1, 3), (2, 4), (5, 6), (5, 7), (7, 9), (6, 8), (8, 10), (5, 11), (6, 12), (11, 12), (11, 13), (13, 15), (12, 14), (14, 16)]
pose = draw_keypoints(img, kout["keypoints"][kk, :, :2].cpu(), connectivity=SKELETON, colors="yellow", radius=5, width=3)
plt.figure(figsize=(6, 8)); plt.imshow(pose.permute(1, 2, 0)); plt.axis("off"); plt.show()

## Extra tasks (optional)

1. **NMS threshold sweep:** run torchvision's NMS on Faster R-CNN's raw boxes (scores > 0.05) with thresholds 0.3, 0.5, 0.7, 0.9 and count the boxes left.
2. **Confidence sweep:** count the kept boxes for confidence 0.1, 0.3, 0.5, 0.7, 0.9. Which knob removes duplicates, which removes weak boxes?
3. **Formats:** convert the kept Faster R-CNN boxes to COCO xywh and to YOLO normalised cxcywh.
4. **Soft-NMS:** instead of deleting overlapping boxes, multiply their score by (1 − IoU). Run it on the six slide boxes.

In [ ]:
# Extra 1 — your code here (hint: nms(out["boxes"].cpu(), out["scores"].cpu(), t))

In [ ]:
# Extra 2 — your code here

In [ ]:
# Extra 3 — your code here

In [ ]:
# Extra 4 — your code here